# ORI 2026 - KOMPLETAN NOTEBOOK V2
## SA FIXOVANIOM NEURONSKOM MREŽOM

Ovaj notebook ima:
- ✅ Očistene CSV fajlove
- ✅ Učitavanje podataka
- ✅ Inženjering indikatora
- ✅ FIXOVANU neuronsku mrežu

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from math import sqrt

warnings.filterwarnings('ignore')

# Auto-pronalaženje CSV foldera
possible_dirs = [".", "data", "./data", "../data"]
INPUT_DIR = "."

for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        csv_files = glob.glob(os.path.join(dir_path, "*_Data.csv"))
        if csv_files and len(csv_files) >= 5:
            INPUT_DIR = dir_path
            break

print(f"✓ CSV fajlovi pronađeni u: {INPUT_DIR}")
print(f"✓ Sve biblioteke učitane!")

✓ CSV fajlovi pronađeni u: data
✓ Sve biblioteke učitane!


## 📋 FAZA 1: Priprema podataka

In [2]:
# Učitaj CSV fajlove
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip().strip('"') for c in df.columns]
    return df

# Pronađi i učitaj sve CSV fajlove
csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_Data.csv")))
print(f"Pronađeno {len(csv_files)} CSV fajlova:\n")

all_data = []
for path in csv_files:
    pair_name = os.path.basename(path).replace("_Data.csv", "")
    df = load_csv(path)
    
    # Procesi datume i cijene
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
    df["Price"] = pd.to_numeric(df["Price"].astype(str).str.replace(",", ""), errors="coerce")
    
    # Uklanja NaN
    df = df[["Date", "Price"]].dropna()
    df["Pair"] = pair_name
    
    all_data.append(df)
    print(f"  ✓ {pair_name:10s}: {len(df)} redova")

# Kombiniraj
currency_all = pd.concat(all_data, ignore_index=True)
currency_all = currency_all.sort_values(["Pair", "Date"]).reset_index(drop=True)

print(f"\n✓ Ukupno: {len(currency_all)} redova za {currency_all['Pair'].nunique()} parova")

Pronađeno 6 CSV fajlova:

  ✓ EURUSD    : 2610 redova
  ✓ USDCAD    : 2610 redova
  ✓ USDCHF    : 2610 redova
  ✓ USDCNY    : 2611 redova
  ✓ USDGBP    : 2610 redova
  ✓ USDJPY    : 2610 redova

✓ Ukupno: 15661 redova za 6 parova


## 🔧 FAZA 2+3: Inženjering indikatora

In [3]:
# Jednostavniji indikatori (bez SAR i CCI koje čine probleme)

def calculate_indicators(price_series):
    """Izračunaj osnovne indikatore"""
    df = pd.DataFrame()
    
    # Osnovni indikatori
    df["SMA_14"] = price_series.rolling(14).mean()
    df["EMA_14"] = price_series.ewm(span=14, adjust=False).mean()
    
    # ROC (Rate of Change)
    df["ROC_14"] = price_series.pct_change(14) * 100
    
    # RSI (Relative Strength Index)
    delta = price_series.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df["RSI_14"] = 100 - (100 / (1 + rs))
    
    # MACD
    ema_12 = price_series.ewm(span=12, adjust=False).mean()
    ema_26 = price_series.ewm(span=26, adjust=False).mean()
    df["MACD"] = ema_12 - ema_26
    df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]
    
    # Standardna devijacija
    df["STD_14"] = price_series.rolling(14).std()
    
    return df

# Primijeni na sve serije
print("Računanje indikatora...\n")

all_features = []
for pair_name in currency_all["Pair"].unique():
    pair_data = currency_all[currency_all["Pair"] == pair_name].sort_values("Date")
    
    # Izračunaj indikatore
    indicators = calculate_indicators(pair_data["Price"].values)
    indicators["Date"] = pair_data["Date"].values
    indicators["Price"] = pair_data["Price"].values
    indicators["Pair"] = pair_name
    
    all_features.append(indicators)
    print(f"  ✓ {pair_name}: {len(indicators)} redova sa 10 indikatora")

features = pd.concat(all_features, ignore_index=True)
features = features[["Pair", "Date", "Price", "SMA_14", "EMA_14", "ROC_14", "RSI_14", "MACD", "MACD_Signal", "MACD_Hist", "STD_14"]]

print(f"\n✓ Shape: {features.shape}")
print(features.head())

AttributeError: 'numpy.ndarray' object has no attribute 'rolling'

## 🧠 FAZA 5+6: FIXOVANA Neuronska Mreža

In [4]:
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)

print("✓ Importovani ML alati")

# Konfiguracija
N_LAGS = 5
TEST_SIZE = 0.2

MLP_PARAMS = {
    'hidden_layer_sizes': (32, 16),
    'activation': 'relu',
    'solver': 'adam',
    'max_iter': 300,
    'early_stopping': True,
    'random_state': 42,
    'verbose': 0
}

print(f"MLP Parametri: {MLP_PARAMS}")

✓ Importovani ML alati
MLP Parametri: {'hidden_layer_sizes': (32, 16), 'activation': 'relu', 'solver': 'adam', 'max_iter': 300, 'early_stopping': True, 'random_state': 42, 'verbose': 0}


In [5]:
def add_lags_and_targets(df, n_lags=5, horizon=1):
    """Dodaj lagged features i target varijable"""
    df_processed = df.copy()
    
    # Dodaj lagged prices
    for lag in range(1, n_lags + 1):
        df_processed[f'Price_lag{lag}'] = df_processed['Price'].shift(lag)
    
    # Dodaj target (cijena u budućnosti)
    df_processed['Target_Price'] = df_processed['Price'].shift(-horizon)
    
    # Dodaj target za klasifikaciju (da li će porast ili pad)
    df_processed['Target_Return'] = df_processed['Target_Price'] - df_processed['Price']
    df_processed['Target_Sign'] = (df_processed['Target_Return'] > 0).astype(int)
    
    # Uklanja NaN
    df_processed = df_processed.dropna()
    
    return df_processed

print("✓ Funkcije za feature engineering definisane")

✓ Funkcije za feature engineering definisane


In [6]:
# Pripremi podatke za eksperimente
results_list = []

for pair_name in features['Pair'].unique():
    print(f"\n{'='*60}")
    print(f"OBRADA: {pair_name}")
    print(f"{'='*60}")
    
    pair_features = features[features['Pair'] == pair_name].copy()
    pair_features = pair_features.sort_values('Date')
    
    # Dodaj features
    pair_processed = add_lags_and_targets(pair_features, n_lags=N_LAGS, horizon=1)
    
    if len(pair_processed) < 100:
        print(f"⚠️  Premalo podataka ({len(pair_processed)})")
        continue
    
    # Definisanja features
    feature_cols = ['SMA_14', 'EMA_14', 'ROC_14', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist', 'STD_14']
    feature_cols += [f'Price_lag{i}' for i in range(1, N_LAGS + 1)]
    feature_cols = [c for c in feature_cols if c in pair_processed.columns]
    
    # Podijeli na train/test
    split_idx = int(len(pair_processed) * (1 - TEST_SIZE))
    
    X_train = pair_processed.iloc[:split_idx][feature_cols].values
    X_test = pair_processed.iloc[split_idx:][feature_cols].values
    
    y_train_reg = pair_processed.iloc[:split_idx]['Target_Price'].values
    y_test_reg = pair_processed.iloc[split_idx:]['Target_Price'].values
    
    y_train_cls = pair_processed.iloc[:split_idx]['Target_Sign'].values
    y_test_cls = pair_processed.iloc[split_idx:]['Target_Sign'].values
    
    print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")
    print(f"Features: {len(feature_cols)}")
    
    # Normalizeuj
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # REGRESIJA - Predikcija cijene
    print("\n📈 REGRESIJA (Predikcija cijene)...")
    try:
        model_reg = MLPRegressor(**MLP_PARAMS)
        model_reg.fit(X_train_scaled, y_train_reg)
        y_pred_reg = model_reg.predict(X_test_scaled)
        
        mae = mean_absolute_error(y_test_reg, y_pred_reg)
        rmse = sqrt(mean_squared_error(y_test_reg, y_pred_reg))
        r2 = r2_score(y_test_reg, y_pred_reg)
        
        print(f"  MAE: {mae:.6f}")
        print(f"  RMSE: {rmse:.6f}")
        print(f"  R²: {r2:.4f}")
        
        results_list.append({
            'Pair': pair_name,
            'Model': 'Regresija',
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'Train_Size': len(X_train),
            'Test_Size': len(X_test)
        })
    except Exception as e:
        print(f"  ❌ Greška: {str(e)[:50]}")
    
    # KLASIFIKACIJA - Predviđanje smjera (gore/dolje)
    print("\n📊 KLASIFIKACIJA (Predviđanje smjera)...")
    try:
        # Provjeri ako imamo obe klase
        if len(np.unique(y_train_cls)) < 2 or len(np.unique(y_test_cls)) < 2:
            print("  ⚠️  Nema dovoljno klasa za obuku")
        else:
            model_cls = MLPClassifier(**MLP_PARAMS)
            model_cls.fit(X_train_scaled, y_train_cls)
            y_pred_cls = model_cls.predict(X_test_scaled)
            
            acc = accuracy_score(y_test_cls, y_pred_cls)
            prec = precision_score(y_test_cls, y_pred_cls, zero_division=0)
            rec = recall_score(y_test_cls, y_pred_cls, zero_division=0)
            f1 = f1_score(y_test_cls, y_pred_cls, zero_division=0)
            
            print(f"  Accuracy: {acc:.4f}")
            print(f"  Precision: {prec:.4f}")
            print(f"  Recall: {rec:.4f}")
            print(f"  F1: {f1:.4f}")
            
            results_list.append({
                'Pair': pair_name,
                'Model': 'Klasifikacija',
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1': f1,
                'Train_Size': len(X_train),
                'Test_Size': len(X_test)
            })
    except Exception as e:
        print(f"  ❌ Greška: {str(e)[:50]}")

print(f"\n{'='*60}")
print(f"✓ Eksperimenti završeni!")
print(f"{'='*60}")

NameError: name 'features' is not defined

## 📊 Analiza rezultata

In [ ]:
# Kreiraj DataFrame sa rezultatima
if results_list:
    results_df = pd.DataFrame(results_list)
    print("\n" + "="*80)
    print("REZULTATI")
    print("="*80 + "\n")
    print(results_df.to_string(index=False))
    
    # Čuva u Excel
    results_df.to_excel('NN_Results.xlsx', index=False)
    print(f"\n✓ Rezultati sačuvani u: NN_Results.xlsx")
else:
    print("❌ Nema rezultata")

## 💾 Čuvanje značajnih fajlova

In [ ]:
# Čuva osnove podatke
currency_all.to_csv('01_Currency_Data_ALL.csv', index=False)
features.to_csv('02_Features_With_Indicators.csv', index=False)

print("✓ Sačuvani fajlovi:")
print("  - 01_Currency_Data_ALL.csv")
print("  - 02_Features_With_Indicators.csv")
print("  - NN_Results.xlsx")